# Batch API: Procesamiento Masivo con 50% de Descuento

> Aprende a procesar miles de requests asíncronamente usando la Batch API
> de Anthropic, con hasta un 50% de descuento sobre el precio estándar.

## Objetivos
- Crear y enviar un batch de clasificación de textos
- Monitorizar el estado del batch con polling
- Descargar y procesar los resultados
- Calcular el ahorro real vs llamadas síncronas

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic pandas --quiet

## 2. Preparar dataset y crear el batch

In [ ]:
import anthropic
import json
import pandas as pd
import time

client = anthropic.Anthropic()

# Dataset de reseñas de productos para clasificar
RESEÑAS = [
    "Producto increíble, superó todas mis expectativas. La calidad es excelente.",
    "Muy decepcionante. Llegó roto y el servicio al cliente no responde.",
    "Cumple su función aunque el precio es algo elevado para lo que ofrece.",
    "¡Genial! Lo recomiendo totalmente, envío rapidísimo y bien embalado.",
    "Regular, ni bueno ni malo. Hace lo que promete pero sin más.",
    "Pésima experiencia. El producto no funciona como se describe en la página.",
    "Calidad sorprendente para el precio. Muy contento con la compra.",
    "El producto es bueno pero el envío tardó 3 semanas, demasiado.",
    "Perfecto para lo que necesitaba. Fácil de usar y resistente.",
    "No lo volvería a comprar. Material barato y se rompió al mes.",
]

# Construir las requests del batch
batch_requests = []
for i, reseña in enumerate(RESEÑAS):
    batch_requests.append({
        "custom_id": f"reseña_{i:03d}",
        "params": {
            "model": "claude-haiku-4-5-20251001",
            "max_tokens": 80,
            "messages": [{
                "role": "user",
                "content": f"""Clasifica esta reseña.
Reseña: \"{reseña}\"
Responde SOLO con JSON: {{\"sentimiento\": \"positivo|negativo|neutro\", \"puntuacion\": 1-5}}"""
            }]
        }
    })

print(f"Preparadas {len(batch_requests)} requests")
print(f"Ejemplo de request:\n{json.dumps(batch_requests[0], indent=2, ensure_ascii=False)}")

## 3. Enviar el batch y monitorizar

In [ ]:
# Enviar el batch
print("Enviando batch...")
batch = client.messages.batches.create(requests=batch_requests)

print(f"✓ Batch creado: {batch.id}")
print(f"  Estado: {batch.processing_status}")
print(f"  Requests: {batch.request_counts}")

# Guardar ID para recuperar en caso de interrupción
with open("batch_id.txt", "w") as f:
    f.write(batch.id)
print(f"\nID guardado en batch_id.txt para recuperación")

# Polling hasta completar
print("\nEsperando resultados (polling cada 10s)...")
intentos = 0
while True:
    intentos += 1
    batch_actual = client.messages.batches.retrieve(batch.id)
    counts = batch_actual.request_counts
    total = counts.processing + counts.succeeded + counts.errored
    print(f"  [{intentos}] {batch_actual.processing_status}: "
          f"✓{counts.succeeded} ✗{counts.errored} ⏳{counts.processing}/{total}")

    if batch_actual.processing_status == "ended":
        print(f"\n✓ Batch completado")
        break

    if intentos > 60:  # máx 10 minutos
        print("Timeout")
        break

    time.sleep(10)

## 4. Descargar y procesar resultados

In [ ]:
resultados = []
errores = []

for resultado in client.messages.batches.results(batch.id):
    custom_id = resultado.custom_id
    idx = int(custom_id.split("_")[1])

    if resultado.result.type == "succeeded":
        texto = resultado.result.message.content[0].text.strip()
        try:
            if "```" in texto:
                texto = texto.split("```")[1].lstrip("json")
            datos = json.loads(texto)
            resultados.append({
                "id": custom_id,
                "reseña": RESEÑAS[idx][:60] + "...",
                "sentimiento": datos.get("sentimiento", "?"),
                "puntuacion": datos.get("puntuacion", 0),
                "tokens": resultado.result.message.usage.output_tokens
            })
        except json.JSONDecodeError:
            errores.append({"id": custom_id, "error": "JSON inválido", "respuesta": texto})
    else:
        errores.append({"id": custom_id, "error": resultado.result.type})

df = pd.DataFrame(resultados)
print("=== RESULTADOS DEL BATCH ===")
print(df.to_string(index=False))
print(f"\nErrores: {len(errores)}")
print(f"\nDistribución de sentimientos:")
print(df["sentimiento"].value_counts())
print(f"\nPuntuación media: {df['puntuacion'].mean():.1f}/5")

## 5. Calcular el ahorro económico

In [ ]:
# Extraer métricas de uso
tokens_salida_total = df["tokens"].sum() if len(df) > 0 else 0
tokens_entrada_estimado = len(batch_requests) * 80  # ~80 tokens por prompt

# Precios Claude Haiku (USD / 1M tokens, 2025)
precio_normal_entrada = 0.25
precio_normal_salida = 1.25
precio_batch_entrada = 0.125   # 50% descuento
precio_batch_salida = 0.625    # 50% descuento

coste_sincrono = (
    tokens_entrada_estimado * precio_normal_entrada +
    tokens_salida_total * precio_normal_salida
) / 1_000_000

coste_batch = (
    tokens_entrada_estimado * precio_batch_entrada +
    tokens_salida_total * precio_batch_salida
) / 1_000_000

print("=== ANÁLISIS DE COSTES ===")
print(f"Requests procesadas: {len(resultados)}")
print(f"Tokens entrada estimados: {tokens_entrada_estimado:,}")
print(f"Tokens salida totales: {tokens_salida_total:,}")
print(f"\nCoste llamadas síncronas: ${coste_sincrono:.6f}")
print(f"Coste Batch API:          ${coste_batch:.6f}")
print(f"Ahorro: ${coste_sincrono - coste_batch:.6f} ({(1 - coste_batch/coste_sincrono)*100:.1f}%)")

# Proyección a escala
print("\n=== PROYECCIÓN A ESCALA ===")
for n in [1_000, 10_000, 100_000, 1_000_000]:
    factor = n / len(RESEÑAS)
    c_sync = coste_sincrono * factor
    c_batch = coste_batch * factor
    print(f"  {n:>9,} reseñas: síncrono ${c_sync:.2f} | batch ${c_batch:.2f} | ahorro ${c_sync-c_batch:.2f}")

# Exportar resultados
df.to_csv("reseñas_clasificadas.csv", index=False, encoding="utf-8")
print("\n✓ Resultados exportados a reseñas_clasificadas.csv")